# Car Mileage Prediction (Regression)
In this notebook, we will analyze the Auto MPG dataset, convert 'MPG' to 'kmpl' (Kilometers Per Liter) to localize the target variable, clean the data, and build a Machine Learning model that predicts replacing mpg.

## Data Science Lifecycle
1. Data Loading
2. Data Cleaning
3. Exploratory Data Analysis
4. Feature Engineering
5. Modeling
6. Evaluation and Export


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Set seaborn design
sns.set_theme(style="whitegrid")


## 1. Data Loading
The dataset `auto-mpg.data` uses varying whitespaces as separators. We will load it specifying the column names manually since they are not in the file.

In [ ]:
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'car_name']
df = pd.read_csv('../data/auto-mpg.data', names=column_names, delim_whitespace=True, na_values="?")
display(df.head())


## 2. Data Cleaning & Feature Engineering
We will:
1. Handle the `?` missing values in `horsepower` by dropping those rows or imputing them. We will stick to dropping them for simplicity here since it's a very tiny percentage of data.
2. We drop `car_name` as it's highly highly cardinal.
3. Convert the target `mpg` to `kmpl` (1 MPG = 0.425144 km/l).

In [ ]:
# Clean missing values
df = df.dropna()
df = df.drop(columns=['car_name'])

# Convert mpg to kmpl
df['kmpl'] = df['mpg'] * 0.425144
df = df.drop(columns=['mpg'])

display(df.head())


## 3. Exploratory Data Analysis (EDA)
Let's see the correlations between features.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(x='weight', y='kmpl', hue='cylinders', data=df)
plt.title("Weight vs KMPL")
plt.show()


## 4. Modeling & Pipeline
Since `origin` is categorical (1=USA, 2=Europe, 3=Japan), we will one-hot encode it. The rest will be scaled.
We will build a Scikit-Learn `Pipeline` which makes our deployment to Django extremely simple.

In [ ]:
# Features and Target
X = df.drop(columns=['kmpl'])
y = df['kmpl']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define preprocessor
numeric_features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

categorical_features = ['origin']
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

# Create Pipeline
rf_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

# Train
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)


## 5. Evaluation & Export
We evaluate the model using Root Mean Squared Error (RMSE) and R-Squared (R2).
Finally, we export the complete pipeline.

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Random Forest RMSE: {rmse:.2f} kmpl")
print(f"Random Forest R^2 : {r2:.2f}")

# Export Pipeline
joblib.dump(rf_model, '../models/car_mileage_rf_model.joblib')
print("Model saved to ../models/car_mileage_rf_model.joblib")
